# Modal

In [ ]:
# | default_exp run_modal

In [ ]:
# | hide
from nbdev.showdoc import *  # type: ignore

Works for now -- in the future make a standalone fastapi server and run on fly gpu.

In [2]:
import os
import subprocess
import sys
from dotenv import load_dotenv  # type: ignore

if os.getenv("USER") == "max":
    # Get the current working directory
    current_dir = os.getcwd()

    # Append the parent directory to sys.path
    parent_dir = os.path.abspath(os.path.join(current_dir, ".."))
    sys.path.append(parent_dir)

    # Decrypt the .env.local file and load variables
    try:
        # Decrypt and output to stdout
        os.system(
            f"dotenvx decrypt -f {os.path.abspath(os.path.join(current_dir, '..'))}/.env.local"
        )

        # Load the decrypted environment variables from stdout
        load_dotenv(
            dotenv_path=os.path.abspath(os.path.join(current_dir, "..")) + "/.env.local"
        )

        os.system(
            f"dotenvx encrypt -f {os.path.abspath(os.path.join(current_dir, '..'))}/.env.local"
        )

        print("Environment variables loaded successfully.")
        os.environ["DOCKER_HOST"] = "unix:///Users/max/.docker/run/docker.sock"
    except subprocess.CalledProcessError as e:
        print(f"Error decrypting .env.local: {e}")
        sys.exit(1)

✔ decrypted (/Users/max/Documents/product_horse/.env.local)
✔ encrypted (/Users/max/Documents/product_horse/.env.local)
Environment variables loaded successfully.


In [3]:
# | export
from product_horse.audio_video import create_video_from_utterances
from product_horse.db import SqlModelDatabase, UtteranceSegment, Video
from product_horse.db import PermissionLevel as DbPermissionLevel
from datetime import datetime, timezone
from jwt.exceptions import InvalidTokenError
from product_horse.filesystems import LocalFileSystem, R2StorageClient
from product_horse.graphql import UtteranceSegmentInput
from uuid import UUID
from random import randint
from pydantic import BaseModel
from fastapi import FastAPI, HTTPException, Header
import os
from typing import Tuple, List, TypedDict, cast
import jwt
import logging

In [47]:
# | export
app = FastAPI()
logger = logging.getLogger(__name__)

api_key = os.getenv("ASSEMBLYAI_API_KEY")
if not api_key:
    raise ValueError("ASSEMBLYAI_API_KEY environment variable not found")

async def run_remote_create_video(
    video_id: UUID,
    utterance_segments_inputs: list[UtteranceSegmentInput],
    employee_id: UUID,
    jwt: str,
    title: str,
    size: Tuple[int, int] = (1920, 1080),
    force_size: bool = True,
) -> Video:
    """
    TO-DOs:
    - recover if file fails to download
    - update video status more immediately and frequently
    - set video failed status if that happens
    """
    db_url = os.getenv("DATABASE_URL")
    db_superuser_url = os.getenv("DATABASE_SUPERUSER_URL")
    if db_superuser_url is None:
        raise Exception("DATABASE_SUPERUSER_URL is not set")
    API_URL = "https://storage.producthorse.workers.dev/"
    if db_url is None:
        raise Exception(
            f"Not Set: {'DATABASE_URL' if db_url is None else ''}"
        )
    database_superuser = SqlModelDatabase(database_url=db_superuser_url)
    database = SqlModelDatabase(database_url=db_url)
    # run on superuser connection, since it's behind jwt auth
    employee = database_superuser.get_employee(employee_id=employee_id)
    if employee is None:
        raise Exception("Employee not found")
    # DEFINE FILE SYSTEMS
    r2 = R2StorageClient(
        api_url=API_URL,
        base_path=str(employee.company_id),
        jwt=jwt,
    )
    server_file_system = LocalFileSystem()
    # rest of program
    utterance_ids = [
        UUID(segment.utterance_id) for segment in utterance_segments_inputs
    ]
    word_ids = [
        word_id
        for segment in utterance_segments_inputs
        for word_id in segment.word_ids
    ]
    utterances = database.as_employee(employee).get_utterances(
        utterance_ids, word_ids=word_ids
    )
    video_exists = database.as_employee(employee).get_all_videos()
    for video in video_exists:
        if video.title == title:  # type: ignore
            title = f"{title}-{randint(1, 200)}"
    utterance_segments_for_video: list[UtteranceSegment] = []
    for utterance in utterances:
        utterance_segments_for_video.append(
            UtteranceSegment(
                id=utterance.id,
                transcription_id=utterance.transcription_id,
                transcription=utterance.transcription,
                words=utterance.words,
                segment_start_word=utterance.words[0],  # type: ignore
                segment_end_word=utterance.words[-1],  # type: ignore
                confidence=utterance.confidence,
                company_id=utterance.company_id,
                created_by_id=utterance.created_by_id,
                start=utterance.start,
                end=utterance.end,
            )
        )
    user = database.as_employee(employee).get_all_users()[0]
    final_destination = await r2.get_unique_file_key(f"{title}.mp4", str(user.id))
    video = None
    try:
        video = await create_video_from_utterances(
            video_id=str(video_id),
            db=database,
            remote_file_system=r2,
            local_file_system=server_file_system,
            employee=employee,
            user=user,
            utterance_segments=utterance_segments_for_video,
            final_destination=final_destination,
            title=title,
            size=size,
            force_size=force_size,
        )
    except Exception as e:
        if "Failed to download file" in str(e):
            video = await create_video_from_utterances(
            video_id=str(video_id),
            db=database,
            remote_file_system=r2,
            local_file_system=server_file_system,
            employee=employee,
            user=user,
            utterance_segments=utterance_segments_for_video,
            final_destination=final_destination,
            title=title,
            size=size,
                force_size=force_size,
            )
        else:
            raise e
    return video

In [ ]:
# | export
class CreateVideoRequest(BaseModel):
    video_id: UUID
    utterance_segments_inputs: List[UtteranceSegmentInput]
    employee_id: UUID
    jwt: str
    title: str
    size: Tuple[int, int] = (1920, 1080)
    force_size: bool = True

class JwtPayload(TypedDict):
    id: UUID
    company_id: UUID
    exp: float
    permission_level: DbPermissionLevel

secret = os.getenv("SECRET")
if not secret:
    raise Exception("JWT_SECRET environment variable not found")

def decode_jwt(token: str) -> JwtPayload:
    payload = cast(JwtPayload, jwt.decode(token, secret, algorithms=["HS256"]))  # type: ignore
    if "id" not in payload:
        raise InvalidTokenError
    if "company_id" not in payload:
        raise InvalidTokenError
    exp_timestamp = datetime.fromtimestamp(payload["exp"], tz=timezone.utc)
    if exp_timestamp < datetime.now(timezone.utc):
        raise InvalidTokenError
    return payload

@app.post("/create_video", response_model=Video)
async def create_video(request: CreateVideoRequest, authorization: str = Header(...)):
    try:
        # Extract the token from the Authorization header
        token = authorization.split("Bearer ")[-1]
        
        # Decode and validate the JWT
        payload = decode_jwt(token)
        
        # Extract employee_id from the decoded JWT
        employee_id = payload["id"]
        
        video = await run_remote_create_video(
            video_id=request.video_id,
            utterance_segments_inputs=request.utterance_segments_inputs,
            employee_id=employee_id,
            jwt=token,
            title=request.title,
            size=request.size,
            force_size=request.force_size
        )
        return video
    except InvalidTokenError:
        logger.error("Invalid or expired token")
        raise HTTPException(status_code=401, detail="Invalid or expired token")
    except Exception as e:
        logger.error(f"Error creating video: {str(e)}")
        raise HTTPException(status_code=500, detail=str(e))

In [ ]:
# | hide
import nbdev

nbdev.nbdev_export()  # type: ignore